#Import

In [ ]:
##Import DA Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

##Import Gdown
import gdown

In [ ]:
##Download file using Gdown
link = 'https://docs.google.com/spreadsheets/d/19cvk9VaGLdt9z2tQm2oWMpS0D1KC7Ba6/edit?usp=drive_link&ouid=117415647210962988644&rtpof=true&sd=true'
gdown.download(link, 'data.xlsx', quiet=False,format='xlsx',fuzzy=True)


Downloading...
From: https://drive.google.com/uc?id=19cvk9VaGLdt9z2tQm2oWMpS0D1KC7Ba6
To: /content/data.xlsx
100%|██████████| 749k/749k [00:00<00:00, 39.7MB/s]


'data.xlsx'

In [ ]:
##Make df from different spreadsheets
sheets = ['raw_sell_in_distributor_a','raw_sell_in_distributor_b','raw_sell_in_distributor_c','raw_sell_out','raw_stock','raw_market_share','raw_product_master']
for sheet in sheets:
    globals()[sheet] = pd.read_excel('data.xlsx', sheet_name=sheet, engine='openpyxl')

In [ ]:
##Download all raw data as CSV
raw_sell_in_distributor_a.to_csv('raw_sell_in_distributor_a.csv')
raw_sell_in_distributor_b.to_csv('raw_sell_in_distributor_b.csv')
raw_sell_in_distributor_c.to_csv('raw_sell_in_distributor_c.csv')
raw_sell_out.to_csv('raw_sell_out.csv')
raw_stock.to_csv('raw_stock.csv')
raw_market_share.to_csv('raw_market_share.csv')
raw_product_master.to_csv('raw_product_master.csv')


In [ ]:
raw_sell_in_distributor_a.head()


,date,store_id,sku_id,qty,value
0,2025-07-05,STORE1,SKU1,4,528360
1,2025-07-09,STORE1,SKU1,2,264180
2,2025-07-10,STORE1,SKU1,1,132090
3,2025-07-11,STORE1,SKU1,1,132090
4,2025-07-12,STORE1,SKU1,2,264180


##Check for Nulls

In [ ]:
for sheet in sheets:
    print(sheet)
    print(globals()[sheet].isnull().sum())

raw_sell_in_distributor_a
date        0
store_id    0
sku_id      0
qty         0
value       0
dtype: int64
raw_sell_in_distributor_b
date        0
store_id    0
sku_id      0
qty         0
value       0
dtype: int64
raw_sell_in_distributor_c
date        0
store_id    0
sku_id      0
qty         0
value       0
dtype: int64
raw_sell_out
date        0
store_id    0
sku_id      0
qty         0
value       0
dtype: int64
raw_stock
date               0
store_id           0
sku_id             0
beginning_stock    0
dtype: int64
raw_market_share
week_date    0
brand        0
format       0
share        0
dtype: int64
raw_product_master
format          0
product_name    0
size            0
sku_id          0
brand           0
price           0
dtype: int64


##Check Duplicated

In [ ]:
for sheet in sheets:
    print(sheet)
    print(globals()[sheet].duplicated(subset=globals()[sheet].iloc[:,0:3]).sum())

raw_sell_in_distributor_a
0
raw_sell_in_distributor_b
0
raw_sell_in_distributor_c
0
raw_sell_out
0
raw_stock
0
raw_market_share
0
raw_product_master
2


In [ ]:
raw_product_master[raw_product_master.duplicated(subset=raw_product_master.iloc[:,0:2],keep=False) == True]

,format,product_name,size,sku_id,brand,price
21,SERUM,ENERGIZING SERUM 25ML,25 ML,SKU22,Venus,120490
22,SERUM,ENERGIZING SERUM 25ML,25 ML,SKU22,Venus,125490
25,SUNSCREEN,TRIPLE PROTECTION SUNSCREEN SPF50 30 ML,30 ML,SKU25,Venus,39700
26,SUNSCREEN,TRIPLE PROTECTION SUNSCREEN SPF50 30 ML,30 ML,SKU25,Venus,36900


We will find the real price by searching their SKU in sells

In [ ]:
##Check for SKU22
raw_sell_in_distributor_a[raw_sell_in_distributor_a['sku_id']=='SKU22']

,date,store_id,sku_id,qty,value
304,2025-07-05,STORE1,SKU22,2,240980
305,2025-07-07,STORE1,SKU22,5,602450
306,2025-07-08,STORE1,SKU22,1,120490
307,2025-07-09,STORE1,SKU22,2,240980
308,2025-07-12,STORE1,SKU22,4,481960
...,...,...,...,...,...
1846,2025-09-15,STORE4,SKU22,1,120490
1847,2025-09-17,STORE4,SKU22,3,361470
1848,2025-09-18,STORE4,SKU22,4,481960
1849,2025-09-29,STORE4,SKU22,5,602450


In [ ]:
##Check for SKU25
raw_sell_in_distributor_a[raw_sell_in_distributor_a['sku_id']=='SKU25']

,date,store_id,sku_id,qty,value
375,2025-07-03,STORE1,SKU25,2,79400
376,2025-07-05,STORE1,SKU25,4,158800
377,2025-07-11,STORE1,SKU25,3,119100
378,2025-07-21,STORE1,SKU25,3,119100
379,2025-07-30,STORE1,SKU25,6,238200
...,...,...,...,...,...
1917,2025-09-20,STORE4,SKU25,2,79400
1918,2025-09-21,STORE4,SKU25,5,198500
1919,2025-09-22,STORE4,SKU25,3,119100
1920,2025-09-25,STORE4,SKU25,4,158800


In [ ]:
79400/2

39700.0

So, the real price for SKU 22 is 120490 and SKU 25 is 39700

In [ ]:
##Dropping row with wrong price
raw_product_master = raw_product_master.drop(index=22)
raw_product_master = raw_product_master.drop(index=26)
raw_product_master.tail(10)

,format,product_name,size,sku_id,brand,price
15,LIQUID SOAP,RELAXING HAIR AND BODY WASH 30 ML,30 ML,SKU16,Venus,7350
16,MOISTURIZER,ACNE MOISTURIZER 30ML,30 ML,SKU17,Venus,7350
17,MOISTURIZER,BRIGHTENING MOISTURIZER 30ML,30 ML,SKU18,Venus,7350
18,MOISTURIZER,ENERGIZING MOISTURIZER 30ML,30 ML,SKU19,Venus,7350
19,SERUM,ACNE SERUM 25ML,25 ML,SKU20,Venus,120490
20,SERUM,BRIGHTENING SERUM 25ML,25 ML,SKU21,Venus,120490
21,SERUM,ENERGIZING SERUM 25ML,25 ML,SKU22,Venus,120490
23,SUNSCREEN,ACNE SUNSCREEN SPF50 30 ML,30 ML,SKU23,Venus,39700
24,SUNSCREEN,BRIGHTENING SUNSCREEN SPF50 30 ML,30 ML,SKU24,Venus,39700
25,SUNSCREEN,TRIPLE PROTECTION SUNSCREEN SPF50 30 ML,30 ML,SKU25,Venus,39700


##Check Type

In [ ]:
for sheet in sheets:
    print(sheet)
    print(globals()[sheet].info())
    print('\n')

raw_sell_in_distributor_a
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2072 entries, 0 to 2071
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      2072 non-null   datetime64[ns]
 1   store_id  2072 non-null   object        
 2   sku_id    2072 non-null   object        
 3   qty       2072 non-null   int64         
 4   value     2072 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 81.1+ KB
None


raw_sell_in_distributor_b
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1546 entries, 0 to 1545
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      1546 non-null   datetime64[ns]
 1   store_id  1546 non-null   object        
 2   sku_id    1546 non-null   object        
 3   qty       1546 non-null   int64         
 4   value     1546 non-null   int64         
dtypes: datetim

In [ ]:
##Making raw_market_share['week_date'] into a datetime
raw_market_share['week_date'] = pd.to_datetime(raw_market_share['week_date'])

#Cleaning

In [ ]:
raw_market_share.head()

,week_date,brand,format,share
0,2025-07-07,brand:Venus,FACIAL WASH,0.290176
1,2025-07-07,brand:Mars,FACIAL WASH,0.248471
2,2025-07-07,brand:Jupiter,FACIAL WASH,0.231506
3,2025-07-07,brand:Saturnus,FACIAL WASH,0.109679
4,2025-07-07,brand:Uranus,FACIAL WASH,0.120167


I want to remove the "brand:"

##Cleaning raw_market_share

In [ ]:
raw_market_share['brand'] = raw_market_share['brand'].str.replace('brand:', '')

In [ ]:
raw_market_share.head()

,week_date,brand,format,share
0,2025-07-07,Venus,FACIAL WASH,0.290176
1,2025-07-07,Mars,FACIAL WASH,0.248471
2,2025-07-07,Jupiter,FACIAL WASH,0.231506
3,2025-07-07,Saturnus,FACIAL WASH,0.109679
4,2025-07-07,Uranus,FACIAL WASH,0.120167


##Making total_sell_in

In [ ]:
##Make for Temp
sell_a = raw_sell_in_distributor_a.copy()
sell_b = raw_sell_in_distributor_b.copy()
sell_c = raw_sell_in_distributor_c.copy()

In [ ]:
##Adding distributor flags to concate later on
raw_sell_in_distributor_a['distributor'] = 'distributor_a'
raw_sell_in_distributor_b['distributor'] = 'distributor_b'
raw_sell_in_distributor_c['distributor'] = 'distributor_c'

raw_sell_in_distributor_a.head()

,date,store_id,sku_id,qty,value,distributor
0,2025-07-05,STORE1,SKU1,4,528360,distributor_a
1,2025-07-09,STORE1,SKU1,2,264180,distributor_a
2,2025-07-10,STORE1,SKU1,1,132090,distributor_a
3,2025-07-11,STORE1,SKU1,1,132090,distributor_a
4,2025-07-12,STORE1,SKU1,2,264180,distributor_a


In [ ]:
##Concating all sell in as sell_in
total_sell_in = pd.concat([raw_sell_in_distributor_a, raw_sell_in_distributor_b, raw_sell_in_distributor_c])
total_sell_in.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5276 entries, 0 to 1657
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         5276 non-null   datetime64[ns]
 1   store_id     5276 non-null   object        
 2   sku_id       5276 non-null   object        
 3   qty          5276 non-null   int64         
 4   value        5276 non-null   int64         
 5   distributor  5276 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 288.5+ KB


In [ ]:
##Concate clean
raw_sell_ins = pd.concat([sell_a, sell_b, sell_c])

##Groupby Date, Store, SKU
grouped_sell_ins = raw_sell_ins.groupby(['date','store_id','sku_id']).agg({'qty':'sum',
                                                                           'value' : 'sum'}).reset_index()
grouped_sell_ins.head()

,date,store_id,sku_id,qty,value
0,2025-07-01,STORE1,SKU14,1,7350
1,2025-07-01,STORE1,SKU3,1,20100
2,2025-07-01,STORE10,SKU1,3,396270
3,2025-07-01,STORE10,SKU13,4,102700
4,2025-07-01,STORE10,SKU14,4,29400


In [ ]:
total_sell_in.head()

,date,store_id,sku_id,qty,value,distributor
0,2025-07-05,STORE1,SKU1,4,528360,distributor_a
1,2025-07-09,STORE1,SKU1,2,264180,distributor_a
2,2025-07-10,STORE1,SKU1,1,132090,distributor_a
3,2025-07-11,STORE1,SKU1,1,132090,distributor_a
4,2025-07-12,STORE1,SKU1,2,264180,distributor_a


In [ ]:
total_sell_in = pd.merge(
                     how='left',
                     left=total_sell_in,
                     right=raw_product_master,
                     left_on=['sku_id'],
                     right_on=['sku_id'])

total_sell_in

,date,store_id,sku_id,qty,value,distributor,format,product_name,size,brand,price
0,2025-07-05,STORE1,SKU1,4,528360,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
1,2025-07-09,STORE1,SKU1,2,264180,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
2,2025-07-10,STORE1,SKU1,1,132090,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
3,2025-07-11,STORE1,SKU1,1,132090,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
4,2025-07-12,STORE1,SKU1,2,264180,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
...,...,...,...,...,...,...,...,...,...,...,...
5271,2025-09-19,STORE9,SKU9,4,140360,distributor_c,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090
5272,2025-09-20,STORE9,SKU9,5,175450,distributor_c,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090
5273,2025-09-22,STORE9,SKU9,4,140360,distributor_c,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090
5274,2025-09-25,STORE9,SKU9,3,105270,distributor_c,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090


In [ ]:
##Reorder columns
total_sell_in = total_sell_in[['date', 'store_id', 'sku_id','distributor','format', 'product_name', 'size', 'brand',  'qty', 'value']]
total_sell_in.head()

,date,store_id,sku_id,distributor,format,product_name,size,brand,qty,value
0,2025-07-05,STORE1,SKU1,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,4,528360
1,2025-07-09,STORE1,SKU1,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,2,264180
2,2025-07-10,STORE1,SKU1,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,1,132090
3,2025-07-11,STORE1,SKU1,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,1,132090
4,2025-07-12,STORE1,SKU1,distributor_a,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,2,264180


##Make raw_stock_merged

In [ ]:
raw_stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23000 entries, 0 to 22999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             23000 non-null  datetime64[ns]
 1   store_id         23000 non-null  object        
 2   sku_id           23000 non-null  object        
 3   beginning_stock  23000 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 718.9+ KB


In [ ]:
raw_sell_out.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4777 entries, 0 to 4776
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      4777 non-null   datetime64[ns]
 1   store_id  4777 non-null   object        
 2   sku_id    4777 non-null   object        
 3   qty       4777 non-null   int64         
 4   value     4777 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 186.7+ KB


In [ ]:
raw_stock_merged = pd.merge(
                     how='left',
                     left=raw_stock,
                     right=raw_sell_out,
                     left_on=['date','store_id','sku_id'],
                     right_on=['date','store_id','sku_id'])

raw_stock_merged = pd.merge(
                     how='left',
                     left=raw_stock_merged,
                     right=grouped_sell_ins,
                     left_on=['date','store_id','sku_id'],
                     right_on=['date','store_id','sku_id'])

raw_stock_merged = raw_stock_merged.fillna(0)
raw_stock_merged = raw_stock_merged.drop(columns=['value_x','value_y'])
raw_stock_merged = raw_stock_merged.rename(columns={'qty_x':'stock_out','qty_y':'stock_in'})
raw_stock_merged['end_stock'] = raw_stock_merged['beginning_stock'] - raw_stock_merged['stock_out'] + raw_stock_merged['stock_in']

##Make rows int
raw_stock_merged['beginning_stock'] = raw_stock_merged['beginning_stock'].astype(int)
raw_stock_merged['stock_out'] = raw_stock_merged['stock_out'].astype(int)
raw_stock_merged['stock_in'] = raw_stock_merged['stock_in'].astype(int)
raw_stock_merged['end_stock'] = raw_stock_merged['end_stock'].astype(int)

raw_stock_merged.head()

,date,store_id,sku_id,beginning_stock,stock_out,stock_in,end_stock
0,2025-07-01,STORE1,SKU1,7,2,0,5
1,2025-07-02,STORE1,SKU1,5,0,0,5
2,2025-07-03,STORE1,SKU1,5,0,0,5
3,2025-07-04,STORE1,SKU1,5,3,0,2
4,2025-07-05,STORE1,SKU1,2,0,4,6


In [ ]:
raw_stock_merged.isnull().sum()

,0
date,0
store_id,0
sku_id,0
beginning_stock,0
stock_out,0
stock_in,0
end_stock,0


In [ ]:
raw_stock_merged = pd.merge(
                     how='left',
                     left=raw_stock_merged,
                     right=raw_product_master,
                     left_on=['sku_id'],
                     right_on=['sku_id'])

raw_stock_merged.head()

,date,store_id,sku_id,beginning_stock,stock_out,stock_in,end_stock,format,product_name,size,brand,price
0,2025-07-01,STORE1,SKU1,7,2,0,5,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
1,2025-07-02,STORE1,SKU1,5,0,0,5,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
2,2025-07-03,STORE1,SKU1,5,0,0,5,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
3,2025-07-04,STORE1,SKU1,5,3,0,2,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
4,2025-07-05,STORE1,SKU1,2,0,4,6,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090


In [ ]:
raw_stock_merged.columns

Index(['date', 'store_id', 'sku_id', 'beginning_stock', 'stock_out',
       'stock_in', 'end_stock', 'format', 'product_name', 'size', 'brand',
       'price'],
      dtype='object')

In [ ]:
##Reorder columns
raw_stock_merged = raw_stock_merged[['date', 'store_id', 'sku_id','format', 'product_name', 'size', 'brand', 'beginning_stock', 'stock_out','stock_in', 'end_stock']]
raw_stock_merged.head()

,date,store_id,sku_id,format,product_name,size,brand,beginning_stock,stock_out,stock_in,end_stock
0,2025-07-01,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,7,2,0,5
1,2025-07-02,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,5,0,0,5
2,2025-07-03,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,5,0,0,5
3,2025-07-04,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,5,3,0,2
4,2025-07-05,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,2,0,4,6


##Cleaning Sell_out

In [ ]:
raw_sell_out.head()

,date,store_id,sku_id,qty,value
0,2025-07-01,STORE1,SKU1,2,132090
1,2025-07-04,STORE1,SKU1,3,132090
2,2025-07-09,STORE1,SKU1,1,132090
3,2025-07-14,STORE1,SKU1,3,132090
4,2025-07-27,STORE1,SKU1,2,132090


In [ ]:
raw_sell_out = raw_sell_out.rename(columns={'value':'price'})
raw_sell_out.head()

,date,store_id,sku_id,qty,price
0,2025-07-01,STORE1,SKU1,2,132090
1,2025-07-04,STORE1,SKU1,3,132090
2,2025-07-09,STORE1,SKU1,1,132090
3,2025-07-14,STORE1,SKU1,3,132090
4,2025-07-27,STORE1,SKU1,2,132090


In [ ]:
raw_sell_out['value'] = raw_sell_out['qty'] * raw_sell_out['price']
raw_sell_out.head()

,date,store_id,sku_id,qty,price,value
0,2025-07-01,STORE1,SKU1,2,132090,264180
1,2025-07-04,STORE1,SKU1,3,132090,396270
2,2025-07-09,STORE1,SKU1,1,132090,132090
3,2025-07-14,STORE1,SKU1,3,132090,396270
4,2025-07-27,STORE1,SKU1,2,132090,264180


In [ ]:
sell_out = pd.merge(
                     how='left',
                     left=raw_sell_out,
                     right=raw_product_master,
                     left_on=['sku_id'],
                     right_on=['sku_id'])

sell_out

,date,store_id,sku_id,qty,price_x,value,format,product_name,size,brand,price_y
0,2025-07-01,STORE1,SKU1,2,132090,264180,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
1,2025-07-04,STORE1,SKU1,3,132090,396270,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
2,2025-07-09,STORE1,SKU1,1,132090,132090,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
3,2025-07-14,STORE1,SKU1,3,132090,396270,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
4,2025-07-27,STORE1,SKU1,2,132090,264180,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,132090
...,...,...,...,...,...,...,...,...,...,...,...
4772,2025-09-13,STORE9,SKU9,2,35090,70180,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090
4773,2025-09-19,STORE9,SKU9,1,35090,35090,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090
4774,2025-09-20,STORE9,SKU9,1,35090,35090,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090
4775,2025-09-25,STORE9,SKU9,2,35090,70180,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,35090


In [ ]:
#reorder columns
sell_out = sell_out[['date', 'store_id', 'sku_id','format', 'product_name', 'size', 'brand', 'qty', 'price_x', 'value']]
sell_out = sell_out.rename(columns={'price_x':'price'})
sell_out.head()

,date,store_id,sku_id,format,product_name,size,brand,qty,price,value
0,2025-07-01,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,2,132090,264180
1,2025-07-04,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,3,132090,396270
2,2025-07-09,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,1,132090,132090
3,2025-07-14,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,3,132090,396270
4,2025-07-27,STORE1,SKU1,FACIAL WASH,ACNE FACE WASH 500 ML,500 ML,Venus,2,132090,264180


In [ ]:
##stock where ending stock = -1
raw_stock_merged[raw_stock_merged['end_stock'] == -1]

,date,store_id,sku_id,format,product_name,size,brand,beginning_stock,stock_out,stock_in,end_stock
1128,2025-07-25,STORE1,SKU20,SERUM,ACNE SERUM 25ML,25 ML,Venus,3,4,0,-1
2035,2025-07-12,STORE1,SKU7,FACIAL WASH,OIL FACE WASH 100 ML,100 ML,Venus,3,4,0,-1
3784,2025-07-13,STORE10,SKU24,SUNSCREEN,BRIGHTENING SUNSCREEN SPF50 30 ML,30 ML,Venus,0,1,0,-1
4560,2025-08-22,STORE10,SKU9,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,8,9,0,-1
4584,2025-09-15,STORE10,SKU9,FACIAL WASH,ENERGIZING FACE WASH 100 ML,100 ML,Venus,0,1,0,-1
5111,2025-08-21,STORE2,SKU14,LIQUID SOAP,ENERGIZING HAIR AND BODY WASH 30 ML,30 ML,Venus,1,2,0,-1
5114,2025-08-24,STORE2,SKU14,LIQUID SOAP,ENERGIZING HAIR AND BODY WASH 30 ML,30 ML,Venus,0,1,0,-1
5692,2025-09-19,STORE2,SKU2,FACIAL WASH,ACNE FACE WASH 100 ML,100 ML,Venus,3,4,0,-1
5710,2025-07-07,STORE2,SKU20,SERUM,ACNE SERUM 25ML,25 ML,Venus,1,2,0,-1
5720,2025-07-17,STORE2,SKU20,SERUM,ACNE SERUM 25ML,25 ML,Venus,2,3,0,-1


#Download

In [ ]:
##Download all as csv
raw_product_master.to_csv('product_master.csv')
raw_stock_merged.to_csv('stock.csv')
raw_market_share.to_csv('market_share.csv')
total_sell_in.to_csv('sell_in.csv')
sell_out.to_csv('sell_out.csv')